# Finite Nuclei — RMF Calculator
Calls the compiled `solver` binary with model parameters and nucleus $(A, N)$, then returns RMS radii and binding energy per nucleon.

In [ ]:
import subprocess
import re
import os
import pandas as pd
import matplotlib.pyplot as plt

SOLVER = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'solver')

In [ ]:
# ── Model parameters (11 values) ─────────────────────────────────────────────
# g2_s, g2_v, g2_rho, alpha, m_s, m_v, m_rho, kappa, lambda, zeta, Lambda
model = [
    110.349189,
    187.694677,
    192.927428,
    3.14159265 * 4 / 137.036,
    447.2455,
    782.50,
    763.0,
    3.260179,
    -0.003551,
    0.0235,
    0.043377,
]

# ── Nucleus ───────────────────────────────────────────────────────────────────
A = 208   # mass number
N = 126   # neutron number  =>  Z = A - N = 82

In [ ]:
def run_rmf(model, A, N):
    """Run solver for nucleus (A, N). Returns {'rms_p', 'rms_n', 'BE'}."""
    cmd = [SOLVER] + [str(p) for p in model] + [str(A), str(N)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError(f'Solver failed (code {result.returncode})')
    for line in result.stdout.splitlines():
        if line.startswith('RESULT:'):
            m = re.search(
                r'rms_p=([\d.eE+\-]+),rms_n=([\d.eE+\-]+),BE=([\d.eE+\-]+)', line)
            if m:
                return {'rms_p': float(m.group(1)),
                        'rms_n': float(m.group(2)),
                        'BE':    float(m.group(3))}
    raise ValueError('RESULT line not found.\n' + result.stdout)

In [ ]:
# ── Single nucleus ────────────────────────────────────────────────────────────
out = run_rmf(model, A, N)
Z   = A - N
print(f'Nucleus       : A={A}, Z={Z}, N={N}')
print(f'RMS proton r  : {out["rms_p"]:.4f} fm')
print(f'RMS neutron r : {out["rms_n"]:.4f} fm')
print(f'Neutron skin  : {out["rms_n"]-out["rms_p"]:.4f} fm')
print(f'B.E./A        : {out["BE"]:.4f} MeV')

In [ ]:
# ── Scan over multiple nuclei ─────────────────────────────────────────────────
nuclei = [
    (16,   8),   # O-16
    (40,  20),   # Ca-40
    (48,  28),   # Ca-48
    (90,  50),   # Zr-90
    (132, 82),   # Sn-132
    (208, 126),  # Pb-208
]
rows = []
for (a, n) in nuclei:
    res = run_rmf(model, a, n)
    rows.append({'A': a, 'Z': a-n, 'N': n,
                 'rms_p (fm)': res['rms_p'],
                 'rms_n (fm)': res['rms_n'],
                 'skin (fm)':  res['rms_n'] - res['rms_p'],
                 'BE/A (MeV)': res['BE']})
    print(f'A={a:3d}  Z={a-n:3d}  done')
df = pd.DataFrame(rows)
df

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['A'], df['BE/A (MeV)'], 'o-', color='steelblue')
axes[0].set_xlabel('Mass number A'); axes[0].set_ylabel('B.E./A (MeV)')
axes[0].set_title('Binding energy per nucleon'); axes[0].grid(True, alpha=0.3)
axes[1].plot(df['A'], df['rms_p (fm)'], 'o-', label='Proton',  color='royalblue')
axes[1].plot(df['A'], df['rms_n (fm)'], 's-', label='Neutron', color='firebrick')
axes[1].set_xlabel('Mass number A'); axes[1].set_ylabel('RMS radius (fm)')
axes[1].set_title('RMS Radii'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()